# Fused-scale ablation evaluation: 0.2"/px vs 0.4"/px (matched from-scratch pair)

**Question.** Does one more rung down in bottleneck scale (v8's 0.4" → 0.2", constant token cost via 128-px crops) buy anything? Runs trained 2026-07-08/09, 300 epochs each, identical recipe (see DOCUMENTATION.md "Fused-scale ablation").

- A = `jaisp_v10_q1_fused02_scratch` (0.2"/px, crop 128, best val 5.71)
- B = `jaisp_v10_q1_fused04_scratch` (0.4"/px, crop 256, best val 6.95 — beats production's 7.10, validating the control)
- Val losses are NOT cross-comparable (different crop statistics) — hence this notebook.

**Protocols.** (1) Masked-band reconstruction on the 39 shared held-out val tiles, identical seeded crops for both models, Fig-3 metric (Pearson r on top-decile info pixels). (2) The frozen Fig-4 decodability probe (RidgeCV + MLP → per-band log-SNR R², source-centered), run at both crop geometries (128 and 256) for both models. Full tiles are infeasible for A (262k tokens), so everything is matched-crop. Outputs → `io/_nb20_outputs/`.

In [1]:
import sys, json, gc, time
from pathlib import Path
import numpy as np, torch

REPO = Path('/home/shemmati/Work/Projects/JAISP')
sys.path.insert(0, str(REPO/'models')); sys.path.insert(0, str(REPO))
from load_foundation import load_foundation
from jaisp_dataset_v10 import JAISPDatasetV10, random_crop_sample
DEV = torch.device('cuda:0')
OUT = REPO/'io'/'_nb20_outputs'
CKPTS = {'A_fused02': REPO/'models/checkpoints/jaisp_v10_q1_fused02_scratch/checkpoint_best.pt',
         'B_fused04': REPO/'models/checkpoints/jaisp_v10_q1_fused04_scratch/checkpoint_best.pt'}
cka = torch.load(CKPTS['A_fused02'], map_location='cpu', weights_only=False)
ckb = torch.load(CKPTS['B_fused04'], map_location='cpu', weights_only=False)
assert cka['val_indices'] == ckb['val_indices']
VAL_IDX = sorted(int(x) for x in cka['val_indices'])
del cka, ckb
DS = JAISPDatasetV10(rubin_dir=str(REPO/'data/rubin_tiles_all'),
                     euclid_dir=str(REPO/'data/euclid_tiles_all_q1'), load_euclid=True, augment=False)
print(len(VAL_IDX), 'shared held-out val tiles')

JAISPDatasetV10: scanning 790 tiles...


  790 tiles passed quality cuts
  790 tiles have Euclid coverage
39 shared held-out val tiles


In [2]:
# ---- (1) masked-band reconstruction, matched seeded crops, Fig-3 metric ----
from jaisp_foundation_v10 import ALL_BANDS
N_CROPS = 2
RES = {}
for tag, ck in CKPTS.items():
    model = load_foundation(str(ck), device=DEV, freeze=True); model.eval()
    for cs in (128, 256):
        key = f'{tag}@crop{cs}'
        t0 = time.time(); rows = []
        for idx in VAL_IDX:
            for ci_ in range(N_CROPS):
                rng = np.random.RandomState(777 + idx*13 + ci_)   # same crops for both models
                s = random_crop_sample(DS[idx], cs, rng)
                pool = {}; pool.update(s['rubin']); pool.update(s.get('euclid', {}))
                if len(pool) < 10: continue
                for band in ALL_BANDS:
                    if band not in pool: continue
                    ctx = [x for x in pool if x != band]
                    cim = {x: pool[x]['image'].unsqueeze(0).to(DEV) for x in ctx}
                    crm = {x: pool[x]['rms'].unsqueeze(0).to(DEV) for x in ctx}
                    ti = pool[band]['image'].unsqueeze(0).to(DEV); tr = pool[band]['rms'].unsqueeze(0).to(DEV)
                    with torch.no_grad():
                        out = model(cim, crm, band, ti, tr)
                    truth = out['target_norm'][0,0].cpu().numpy(); pred = out['pred'][0,0].cpu().numpy()
                    info = out['info_weights'][0,0].cpu().numpy()
                    m = info >= np.nanpercentile(info, 90)
                    if m.sum() < 20: continue
                    r = float(np.corrcoef(truth[m], pred[m])[0,1])
                    rows.append(dict(idx=idx, crop=ci_, band=band, r=r))
        RES[key] = rows
        med = np.median([x['r'] for x in rows])
        print(f'{key}: {len(rows)} recons, overall median r={med:.4f}  ({time.time()-t0:.0f}s)')
    del model; gc.collect(); torch.cuda.empty_cache()
json.dump(RES, open(OUT/'recon_matched_crops.json','w'))

JAISPFoundationV10: 6.5M trainable parameters
  stem_ch=64, hidden_ch=256, fused_scale=0.20"/px
  stream_depths={'rubin': 1, 'euclid': 1}
  rubin_concat=True  (symmetric concat fusion)
  loss_type=charbonnier  charbonnier_eps=0.001  core_l2_weight=0.2  core_info_thr=0.5
  rubin decoder channels: [128]
  euclid decoder channels: [128]


Loaded v10 foundation (fused_scale=0.2, rubin_concat=True, loss=charbonnier, core_l2=0.2) from /home/shemmati/Work/Projects/JAISP/models/checkpoints/jaisp_v10_q1_fused02_scratch/checkpoint_best.pt


A_fused02@crop128: 780 recons, overall median r=0.8519  (138s)


A_fused02@crop256: 780 recons, overall median r=0.8976  (1018s)


JAISPFoundationV10: 9.2M trainable parameters
  stem_ch=64, hidden_ch=256, fused_scale=0.40"/px
  stream_depths={'rubin': 1, 'euclid': 2}
  rubin_concat=True  (symmetric concat fusion)
  loss_type=charbonnier  charbonnier_eps=0.001  core_l2_weight=0.2  core_info_thr=0.5
  rubin decoder channels: [128]
  euclid decoder channels: [256, 128]
Loaded v10 foundation (fused_scale=0.4, rubin_concat=True, loss=charbonnier, core_l2=0.2) from /home/shemmati/Work/Projects/JAISP/models/checkpoints/jaisp_v10_q1_fused04_scratch/checkpoint_best.pt


B_fused04@crop128: 780 recons, overall median r=0.8568  (75s)


B_fused04@crop256: 780 recons, overall median r=0.8982  (163s)


In [3]:
# per-band medians table
import collections
def band_med(rows):
    d = collections.defaultdict(list)
    for x in rows: d[x['band']].append(x['r'])
    return {b: float(np.median(v)) for b, v in d.items()}
TAB = {k: band_med(v) for k, v in RES.items()}
bands = ALL_BANDS
hdr = f"{'band':<12}" + ''.join(f"{k:>18}" for k in TAB)
print(hdr); print('-'*len(hdr))
for b in bands:
    print(f"{b:<12}" + ''.join(f"{TAB[k].get(b, float('nan')):>18.4f}" for k in TAB))
for lbl, sel in [('RUBIN med', [b for b in bands if not b.startswith('euclid')]),
                 ('EUCLID med', [b for b in bands if b.startswith('euclid')])]:
    print(f"{lbl:<12}" + ''.join(f"{np.median([TAB[k][b] for b in sel if b in TAB[k]]):>18.4f}" for k in TAB))

band         A_fused02@crop128 A_fused02@crop256 B_fused04@crop128 B_fused04@crop256
------------------------------------------------------------------------------------
rubin_u                 0.1865            0.2672            0.1850            0.2655
rubin_g                 0.9636            0.9839            0.9599            0.9867
rubin_r                 0.9857            0.9902            0.9867            0.9925
rubin_i                 0.9837            0.9900            0.9851            0.9910
rubin_z                 0.9395            0.9658            0.9393            0.9656
rubin_y                 0.4520            0.6423            0.4599            0.6425
euclid_VIS              0.6326            0.7167            0.6384            0.7176
euclid_Y                0.7448            0.8338            0.7443            0.8328
euclid_J                0.8106            0.8733            0.8122            0.8729
euclid_H                0.8564            0.8935            0.854

In [4]:
# ---- (2) frozen Fig-4 decodability probe, both models x both crop sizes ----
import torch.nn as nn
from scipy.ndimage import gaussian_filter, maximum_filter
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from jaisp_foundation_v10 import RUBIN_BANDS, EUCLID_BANDS, band_group, STREAM_PIXEL_SCALES
from models.older_architectures.jaisp_dataset_v6 import JAISPDatasetV6
from models.older_architectures.jaisp_dataset_v8 import random_crop_sample as rcs8

N_TILES=60; KFOLD=6; SEED=13013
SNR_THR=5.0; MIN_DIST=8; MIN_TEST=8; MIN_TRAIN=30; MAX_PEAKS=2000
TARGET_FLOOR=-0.5; ALPHAS=np.logspace(-1,4,12)

ds6 = JAISPDatasetV6(rubin_dir=str(REPO/'data/rubin_tiles_all'),
                     euclid_dir=str(REPO/'data/euclid_tiles_all_q1'), load_euclid=True, augment=False)
sel=[]
for idx in range(len(ds6)):
    if len(sel)>=N_TILES: break
    s=ds6[idx]
    if s['has_euclid'] and all(b in (set(s['rubin'])|set(s['euclid'])) for b in ALL_BANDS): sel.append(idx)

def bdicts(s):
    im,rm={},{}
    for g in ('rubin','euclid'):
        for b,d in s[g].items():
            im[b]=d['image'].to(DEV).unsqueeze(0).float(); rm[b]=d['rms'].to(DEV).unsqueeze(0).float()
    return im,rm
def peaks(vi,vr):
    sm=gaussian_filter(vi/np.maximum(vr,1e-9),1.5); lm=maximum_filter(sm,size=MIN_DIST)==sm
    ys,xs=np.where(lm&(sm>SNR_THR)&(vr>0))
    if len(ys)>MAX_PEAKS:
        o=np.argsort(sm[ys,xs])[::-1][:MAX_PEAKS]; ys,xs=ys[o],xs[o]
    return ys,xs
class MLP(nn.Module):
    def __init__(s,d,h=128):
        super().__init__(); s.net=nn.Sequential(nn.Linear(d,h),nn.GELU(),nn.Dropout(0.1),nn.Linear(h,h),nn.GELU(),nn.Linear(h,1))
    def forward(s,x): return s.net(x).squeeze(-1)
def train_mlp(Xtr,ytr,Xte,yte,epochs=120):
    Xt=torch.from_numpy(Xtr).float().to(DEV); yt=torch.from_numpy(ytr).float().to(DEV); Xe=torch.from_numpy(Xte).float().to(DEV)
    m=MLP(Xtr.shape[1]).to(DEV); opt=torch.optim.AdamW(m.parameters(),lr=2e-3,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs); best=-1e9
    for _ in range(epochs):
        m.train(); perm=torch.randperm(len(Xt),device=DEV)
        for i in range(0,len(Xt),256):
            sl=perm[i:i+256]; loss=((m(Xt[sl])-yt[sl])**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
        sch.step(); m.eval()
        with torch.no_grad(): p=m(Xe).cpu().numpy()
        best=max(best,1.0-((yte-p)**2).sum()/max(((yte-yte.mean())**2).sum(),1e-9))
    return float(best)

def run_probe(ckpt_path, crop_size, tag):
    model = load_foundation(str(ckpt_path), device=DEV, freeze=True); model.eval()
    fused = model.encoder.fused_pixel_scale_arcsec
    @torch.no_grad()
    def source_features(s):
        im,rm=bdicts(s); bn=model.encode(context_images=im,context_rms=rm)['bottleneck']
        _,C,Hb,Wb=bn.shape; vi=im['euclid_VIS'][0,0].cpu().numpy(); vr=rm['euclid_VIS'][0,0].cpu().numpy()
        ys,xs=peaks(vi,vr)
        by=(ys*STREAM_PIXEL_SCALES['euclid']/fused).astype(int); bx=(xs*STREAM_PIXEL_SCALES['euclid']/fused).astype(int)
        k=(by>=4)&(by<Hb-4)&(bx>=4)&(bx<Wb-4); by,bx=by[k],bx[k]
        if len(by)==0: return np.zeros((0,C),np.float32),{}
        feats=bn[0,:,by,bx].T.cpu().numpy().astype(np.float32); tg={}
        for b in ALL_BANDS:
            nb_=STREAM_PIXEL_SCALES[band_group(b)]; r=fused/nb_
            yn=np.clip((by*r).astype(int),2,im[b].shape[-2]-3); xn=np.clip((bx*r).astype(int),2,im[b].shape[-1]-3)
            it=im[b][0,0].cpu().numpy(); rt=rm[b][0,0].cpu().numpy(); v=np.zeros(len(by),np.float32)
            for j in range(len(by)):
                y,x=yn[j],xn[j]; v[j]=np.log10(max(it[y-2:y+3,x-2:x+3].mean()/(rt[y-2:y+3,x-2:x+3].mean()+1e-9),1e-3))
            tg[b]=v
        return feats,tg
    torch.manual_seed(0); np.random.seed(0)
    F,Tg,tid=[],{b:[] for b in ALL_BANDS},[]
    for ti,idx in enumerate(sel):
        f,tg=source_features(rcs8(ds6[idx], crop_size, np.random.RandomState(SEED+int(idx)*1009+int(ti))))
        if len(f)==0: continue
        F.append(f); [Tg[b].append(v) for b,v in tg.items()]; tid.append(np.full(len(f),ti,np.int32))
    X=np.concatenate(F); Y={b:np.concatenate(v) for b,v in Tg.items()}; T=np.concatenate(tid)
    uniq=list(np.unique(T)); fold=np.array([uniq.index(t)%KFOLD for t in T])
    rg,mp={},{}
    for b in ALL_BANDS:
        y=Y[b]; m=y>TARGET_FLOOR; Xb,yb,fb=X[m],y[m],fold[m]; rf,mf=[],[]
        for k in range(KFOLD):
            te=fb==k; tr=~te
            if te.sum()<MIN_TEST or tr.sum()<MIN_TRAIN: continue
            sc=StandardScaler().fit(Xb[tr]); Xtr=sc.transform(Xb[tr]).astype(np.float32); Xte=sc.transform(Xb[te]).astype(np.float32)
            rf.append(r2_score(yb[te], RidgeCV(alphas=ALPHAS).fit(Xtr,yb[tr]).predict(Xte)))
            mf.append(train_mlp(Xtr,yb[tr],Xte,yb[te]))
        rg[b]=float(np.mean(rf)); mp[b]=float(np.mean(mf))
    res=dict(checkpoint=str(ckpt_path), tag=tag, fused_scale=float(fused), crop=int(crop_size),
             n_sources=int(X.shape[0]), ridge_r2=rg, mlp_r2=mp)
    json.dump(res, open(OUT/f'probe_{tag}.json','w'), indent=1)
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f'probe_{tag}: n={X.shape[0]}, Rubin MLP R2={np.mean([mp[b] for b in RUBIN_BANDS]):.3f}, '
          f'Euclid MLP R2={np.mean([mp[b] for b in EUCLID_BANDS]):.3f}')
    return res

PROBES={}
for tag, ck in CKPTS.items():
    for cs in (256, 128):
        t0=time.time(); PROBES[f'{tag}@crop{cs}']=run_probe(ck, cs, f'{tag}_crop{cs}'); print(f'  ({time.time()-t0:.0f}s)')

JAISPDatasetV6: scanning 790 tiles...


  790 tiles passed quality cuts
  790 tiles have Euclid coverage


JAISPFoundationV10: 6.5M trainable parameters
  stem_ch=64, hidden_ch=256, fused_scale=0.20"/px
  stream_depths={'rubin': 1, 'euclid': 1}
  rubin_concat=True  (symmetric concat fusion)
  loss_type=charbonnier  charbonnier_eps=0.001  core_l2_weight=0.2  core_info_thr=0.5
  rubin decoder channels: [128]
  euclid decoder channels: [128]
Loaded v10 foundation (fused_scale=0.2, rubin_concat=True, loss=charbonnier, core_l2=0.2) from /home/shemmati/Work/Projects/JAISP/models/checkpoints/jaisp_v10_q1_fused02_scratch/checkpoint_best.pt


probe_A_fused02_crop256: n=815, Rubin MLP R2=0.930, Euclid MLP R2=0.899
  (158s)
JAISPFoundationV10: 6.5M trainable parameters
  stem_ch=64, hidden_ch=256, fused_scale=0.20"/px
  stream_depths={'rubin': 1, 'euclid': 1}
  rubin_concat=True  (symmetric concat fusion)
  loss_type=charbonnier  charbonnier_eps=0.001  core_l2_weight=0.2  core_info_thr=0.5
  rubin decoder channels: [128]
  euclid decoder channels: [128]
Loaded v10 foundation (fused_scale=0.2, rubin_concat=True, loss=charbonnier, core_l2=0.2) from /home/shemmati/Work/Projects/JAISP/models/checkpoints/jaisp_v10_q1_fused02_scratch/checkpoint_best.pt


probe_A_fused02_crop128: n=217, Rubin MLP R2=0.814, Euclid MLP R2=0.781
  (67s)
JAISPFoundationV10: 9.2M trainable parameters
  stem_ch=64, hidden_ch=256, fused_scale=0.40"/px
  stream_depths={'rubin': 1, 'euclid': 2}
  rubin_concat=True  (symmetric concat fusion)
  loss_type=charbonnier  charbonnier_eps=0.001  core_l2_weight=0.2  core_info_thr=0.5
  rubin decoder channels: [128]
  euclid decoder channels: [256, 128]
Loaded v10 foundation (fused_scale=0.4, rubin_concat=True, loss=charbonnier, core_l2=0.2) from /home/shemmati/Work/Projects/JAISP/models/checkpoints/jaisp_v10_q1_fused04_scratch/checkpoint_best.pt


probe_B_fused04_crop256: n=767, Rubin MLP R2=0.927, Euclid MLP R2=0.891
  (90s)
JAISPFoundationV10: 9.2M trainable parameters
  stem_ch=64, hidden_ch=256, fused_scale=0.40"/px
  stream_depths={'rubin': 1, 'euclid': 2}
  rubin_concat=True  (symmetric concat fusion)
  loss_type=charbonnier  charbonnier_eps=0.001  core_l2_weight=0.2  core_info_thr=0.5
  rubin decoder channels: [128]
  euclid decoder channels: [256, 128]
Loaded v10 foundation (fused_scale=0.4, rubin_concat=True, loss=charbonnier, core_l2=0.2) from /home/shemmati/Work/Projects/JAISP/models/checkpoints/jaisp_v10_q1_fused04_scratch/checkpoint_best.pt


probe_B_fused04_crop128: n=193, Rubin MLP R2=0.695, Euclid MLP R2=0.820
  (63s)


In [5]:
# ---- summary: recon + probe side by side, with production q1_long reference ----
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
prod = json.load(open(REPO/'io/_nb13_outputs/cross_instrument_summary_v10_fixed.json'))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
bands = ALL_BANDS; ypos = np.arange(len(bands))
# panel 1: recon per-band median r at crop128 (matched, A's home geometry)
for k, col in [('A_fused02@crop128','#d62728'), ('B_fused04@crop128','#1f77b4')]:
    axes[0].plot([TAB[k].get(b) for b in bands], ypos, 'o-', color=col, label=k)
axes[0].set_title('masked-band recon median r (crop 128)')
# panel 2: recon at crop256 (frozen-protocol geometry, B's home)
for k, col in [('A_fused02@crop256','#d62728'), ('B_fused04@crop256','#1f77b4')]:
    axes[1].plot([TAB[k].get(b) for b in bands], ypos, 'o-', color=col, label=k)
axes[1].set_title('masked-band recon median r (crop 256)')
# panel 3: probe MLP R2 at crop256 vs production
for k, col in [('A_fused02@crop256','#d62728'), ('B_fused04@crop256','#1f77b4')]:
    axes[2].plot([PROBES[k]['mlp_r2'][b] for b in bands], ypos, 'o-', color=col, label=k)
axes[2].plot([prod['diag2b_source_mlp_r2'][b] for b in bands], ypos, 's--', color='0.4', label='production q1_long')
axes[2].set_title('decodability probe MLP R$^2$ (crop 256)')
for ax in axes:
    ax.set_yticks(ypos); ax.set_yticklabels(bands); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT/'fused_scale_ablation_summary.png', dpi=150)
print('saved', OUT/'fused_scale_ablation_summary.png')

def agg(d, sel): return float(np.mean([d[b] for b in sel]))
print('\n=== VERDICT INPUTS ===')
for cs in (128, 256):
    a, b = TAB[f'A_fused02@crop{cs}'], TAB[f'B_fused04@crop{cs}']
    print(f'recon@{cs}: A allband-med {np.median(list(a.values())):.4f} vs B {np.median(list(b.values())):.4f}')
for cs in (128, 256):
    a, b = PROBES[f'A_fused02@crop{cs}'], PROBES[f'B_fused04@crop{cs}']
    for grp, sel in [('Rubin', RUBIN_BANDS), ('Euclid', EUCLID_BANDS)]:
        print(f'probe@{cs} {grp} MLP R2: A {agg(a["mlp_r2"], sel):.3f} vs B {agg(b["mlp_r2"], sel):.3f}')

saved /home/shemmati/Work/Projects/JAISP/io/_nb20_outputs/fused_scale_ablation_summary.png

=== VERDICT INPUTS ===
recon@128: A allband-med 0.8335 vs B 0.8335
recon@256: A allband-med 0.8834 vs B 0.8843
probe@128 Rubin MLP R2: A 0.814 vs B 0.695
probe@128 Euclid MLP R2: A 0.781 vs B 0.820
probe@256 Rubin MLP R2: A 0.930 vs B 0.927
probe@256 Euclid MLP R2: A 0.899 vs B 0.891
